# DeepWeeds Lab Day 2
Nguyen Hong Cuong - 2A202602415. Select every decision on validation; enable test export only after the final configuration is frozen.

In [ ]:
%pip -q install timm openpyxl scikit-learn matplotlib
import os, sys, platform, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

REPO_DIR = Path.cwd()  # change only if the notebook kernel starts elsewhere
CODE_DIR = REPO_DIR / 'submissions/2A202602415_nguyen_hong_cuong/code'
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(CODE_DIR))
print('python', platform.python_version(), 'torch', torch.__version__, 'timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## Data integrity
The official archive MD5 is `b7b30f96d466fba86016aa5a26606e0f`. Keep the archive until this check passes.

In [ ]:
archive = REPO_DIR / 'images.zip'
if archive.exists():
    actual = hashlib.md5(archive.read_bytes()).hexdigest()
    assert actual == 'b7b30f96d466fba86016aa5a26606e0f', actual
    print('MD5 OK')
else:
    print('images.zip is absent; verify MD5 immediately after download before extracting')

IMAGES_DIR = REPO_DIR / 'images'
LABELS_DIR = REPO_DIR / 'data/labels'

In [ ]:
from dataset import load_split, check_split
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_stats = check_split(train_df, val_df, test_df, IMAGES_DIR)
display(pd.DataFrame(split_stats['per_class']).rename_axis('Label'))
pd.DataFrame(split_stats['per_class']).plot.bar(figsize=(10, 4), title='DeepWeeds fold 0 class distribution')

## Backbone screen and controlled ablations

In [ ]:
from experiments import registry
backbone_configs = registry('backbones', backbone='convnext_tiny', seed=0)
[(c.exp_id, c.backbone) for c in backbone_configs]
# GPU run: from train import run; [run(c) for c in backbone_configs]

In [ ]:
# Replace with the backbone selected from validation results.
selected_backbone = 'convnext_tiny'
ablation_configs = registry('ablations', backbone=selected_backbone, seed=0)
[(c.exp_id, c.init, c.aug, c.loss, c.mix, c.ema_decay) for c in ablation_configs]
# GPU run: [run(c) for c in ablation_configs]

## Inference experiments
Use `inference.py` for flip/multicrop/multiscale TTA, probability or logit aggregation, ensembles, temperature scaling fitted only on validation, and Conv-BN fusion. Use `benchmark.py` for synchronized p50/p95/p99 latency.

In [ ]:
from inference import aggregate_views, fit_temperature, apply_temperature
from benchmark import latency_report
# Example after loading saved validation logits:
# T = fit_temperature(val_logits, val_labels)
# calibrated_test_probs = apply_temperature(test_logits, T)

## Final test protocol
Freeze the full configuration from validation. Run F01 and baseline T00 for seeds 0, 1, and 2 with `save_test_predictions=True`. Never revise the configuration after viewing test metrics.

In [ ]:
# Example only after validation selection:
# from train import Config, run
# for seed in (0, 1, 2):
#     run(Config(exp_id='F01', seed=seed, backbone=selected_backbone,
#                save_test_predictions=True, ...))
#     run(Config(exp_id='T00', seed=seed, backbone=selected_backbone,
#                save_test_predictions=True))

In [ ]:
# Run from the repository root after final predictions exist:
# !python eval.py score --pred 'submissions/2A202602415_nguyen_hong_cuong/predictions/F01_seed*_test.csv' --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out
# !python eval.py grade --final 'submissions/2A202602415_nguyen_hong_cuong/predictions/F01_seed*_test.csv' --baseline 'submissions/2A202602415_nguyen_hong_cuong/predictions/T00_seed*_test.csv' --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --out eval_out